In [14]:
import sys
sys.path.append("..")
from src.db import query_cached, append_note
from src.cleaning import detect_contaminated_devices, filter_non_laptop_devices, normalize_laptop_weight
import pandas as pd
from datetime import datetime
import os

df_analysis = pd.read_csv("../data/processed/analysis_base.csv")

In [15]:
### [Giai đoạn 5 - Trang 1] Overview stats cho Manager
overview_query = """
SELECT
    COUNT(DISTINCT user_psuedo_id) AS total_active_users,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(*) AS total_events
FROM laplaptech.user_event_tracking
"""
df_overview = query_cached(overview_query, cache_dir="../data/raw", force_refresh=True)
display(df_overview)

# Phân bố "người dùng" theo phân khúc máy: dùng interest_score/pageview_count (mức độ chú ý
# thực tế của người dùng), KHÔNG chỉ đếm số lượng laptop trong từng phân khúc.
def segment_label(row):
    if row["is_gaming_laptop"] == True:
        return "Gaming"
    if row["is_workstation"] == True:
        return "Workstation"
    return "Office"

df_analysis["segment"] = df_analysis.apply(segment_label, axis=1)
df_segment_attention = df_analysis.groupby("segment").agg(
    laptop_count=("id", "count"),
    total_pageview=("pageview_count", "sum"),
    total_interest_score=("interest_score", "sum"),
).reset_index()
display(df_segment_attention)

df_overview.to_csv("../data/processed/dashboard_overview_kpi.csv", index=False)
df_segment_attention.to_csv("../data/processed/dashboard_segment_attention.csv", index=False)
print("Đã lưu: dashboard_overview_kpi.csv, dashboard_segment_attention.csv")
print("Top laptop dùng trực tiếp analysis_base.csv (sort theo interest_score trong Power BI), không cần file riêng.")

,total_active_users,total_sessions,total_events
0,34887,87353,774312


,segment,laptop_count,total_pageview,total_interest_score
0,Gaming,47,30512,46274
1,Office,86,72010,113378
2,Workstation,2,1818,2694


Đã lưu: dashboard_overview_kpi.csv, dashboard_segment_attention.csv
Top laptop dùng trực tiếp analysis_base.csv (sort theo interest_score trong Power BI), không cần file riêng.


In [16]:
### [Giai đoạn 5 - Trang 2] CPU/GPU battery ratio — flag rõ độ thưa, dùng battery_density thay thế
cpu_gpu_query = """
SELECT
    c.name AS cpu_name,
    b.geekbench_6_cpu_multi_core_plugged_in,
    b.geekbench_6_cpu_multi_core_battery,
    b.laptop_model_id
FROM laplaptech.laptop_benchmark_result b
INNER JOIN laplaptech.laptop_model m ON b.laptop_model_id = m.id
INNER JOIN laplaptech.cpu_model c ON m.cpu_model_id = c.id
WHERE b.geekbench_6_cpu_multi_core_plugged_in IS NOT NULL
  AND b.geekbench_6_cpu_multi_core_battery IS NOT NULL
"""
df_cpu_gpu = query_cached(cpu_gpu_query, cache_dir="../data/raw", force_refresh=True)
df_cpu_gpu["cpu_battery_ratio"] = (
    df_cpu_gpu["geekbench_6_cpu_multi_core_battery"] / df_cpu_gpu["geekbench_6_cpu_multi_core_plugged_in"]
)
print(f"Số dòng có đủ dữ liệu để tính CPU_Battery_Ratio: {len(df_cpu_gpu)}")
df_cpu_gpu.to_csv("../data/processed/dashboard_cpu_battery_ratio.csv", index=False)

# battery_density theo hãng CPU — chỉ số thay thế có độ phủ tốt hơn (population đã lọc sạch)
df_laptop_raw = query_cached("SELECT * FROM laplaptech.laptop_model", cache_dir="../data/raw")
df_cpu_names = query_cached("SELECT id, name FROM laplaptech.cpu_model", cache_dir="../data/raw")
df_laptop_clean = filter_non_laptop_devices(df_laptop_raw)
df_laptop_clean = df_laptop_clean.merge(df_cpu_names, left_on="cpu_model_id", right_on="id",
                                          suffixes=("", "_cpu"))
df_laptop_clean["laptop_weight_g"] = normalize_laptop_weight(df_laptop_clean)
df_laptop_clean["battery_density"] = pd.to_numeric(
    df_laptop_clean["battery_capacity_whr"], errors="coerce"
) / df_laptop_clean["laptop_weight_g"].replace(0, pd.NA)

df_density_by_cpu = df_laptop_clean.groupby("name_cpu")["battery_density"].agg(
    ["count", "mean", "median"]
).reset_index().sort_values("count", ascending=False)
display(df_density_by_cpu)
df_density_by_cpu.to_csv("../data/processed/dashboard_battery_density_by_cpu.csv", index=False)

note_page2 = f"""
## [Giai đoạn 5 - Trang 2] Ghi chú độ khả dụng dữ liệu

CPU_Battery_Ratio: chỉ {len(df_cpu_gpu)} dòng đủ dữ liệu Geekbench -> KHÔNG dùng làm biểu đồ
chính (đã xuất file để minh bạch). Thay thế: dashboard_battery_density_by_cpu.csv, độ phủ tốt
hơn nhiều, dùng cùng Scatter Trọng lượng vs Battery Density đã có trong analysis_base.csv.
"""
append_note(note_page2)

Số dòng có đủ dữ liệu để tính CPU_Battery_Ratio: 1


,name_cpu,count,mean,median
35,Intel Core Ultra 7 258V,9,0.054076,0.056062
7,AMD Ryzen AI 7 350,6,0.051640,0.051000
30,Intel Core Ultra 7 155H,5,0.045290,0.048993
11,AMD Ryzen AI 9 HX 370,5,0.052464,0.051714
29,Intel Core Ultra 5 226V,4,0.050685,0.051546
...,...,...,...,...
59,Intel Core i9-14900HX,1,0.043975,0.043975
66,Qualcomm Snapdragon X2 Elite X2E 88 100,1,0.070707,0.070707
63,Qualcomm Snapdragon X Plus X1P42100,1,0.038514,0.038514
18,Apple M3 Ultra,0,NaN,NaN


In [17]:
### [Giai đoạn 5 - Trang 3] Funnel: Tìm kiếm -> Xem chi tiết -> Chọn So sánh
# Endpoint đã đổi so với kế hoạch gốc: không có event "xem video review" trong dữ liệu
# (đã xác nhận ở Giai đoạn 3), nên funnel dừng ở "Chọn So sánh".
funnel_query = """
SELECT
    'Tim kiem' AS stage, COUNT(DISTINCT session_id) AS session_count, 1 AS stage_order
FROM laplaptech.user_event_tracking WHERE event_name = 'search_for_device'
UNION ALL
SELECT
    'Xem chi tiet', COUNT(DISTINCT session_id), 2
FROM laplaptech.user_event_tracking
WHERE event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'
UNION ALL
SELECT
    'Chon So sanh', COUNT(DISTINCT session_id), 3
FROM laplaptech.user_event_tracking WHERE event_name = 'select_device_for_comparison'
"""
df_funnel = query_cached(funnel_query, cache_dir="../data/raw", force_refresh=True)
df_funnel = df_funnel.sort_values("stage_order")
display(df_funnel)
df_funnel.to_csv("../data/processed/dashboard_funnel_summary.csv", index=False)

note_page3 = """
## [Giai đoạn 5 - Trang 3] Ghi chú funnel

Funnel đếm số session PHÂN BIỆT có xuất hiện event ở mỗi bước (không bắt buộc thứ tự thời gian
trong cùng session) — là ước lượng mức độ tham gia từng bước, không phải phễu chuyển đổi
tuyệt đối theo trình tự. Endpoint đổi từ "click video review" (kế hoạch gốc) sang
"select_device_for_comparison", do không có dữ liệu tracking cho hành vi xem video.
"""
append_note(note_page3)

,stage,session_count,stage_order
0,Tim kiem,37516,1
2,Xem chi tiet,44159,2
1,Chon So sanh,8482,3


In [18]:
### [Giai đoạn 5 - Trang 3] Điều tra hiện tượng không đơn điệu giảm + kết luận CHÍNH THỨC
# (Lịch sử đầy đủ của quá trình điều tra — bao gồm bước từng overclaim rồi tự đính chính —
#  đã được lưu vĩnh viễn trong project_notes.md từ các lần chạy trước. Cell này chỉ giữ lại
#  logic tính toán và kết luận ĐÃ ĐÚNG, tránh lặp lại việc ghi đè note sai mỗi lần re-run.)

search_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking WHERE event_name = 'search_for_device'",
    cache_dir="../data/raw", force_refresh=True)["session_id"])
detail_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking "
    "WHERE event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'DeviceDetail'",
    cache_dir="../data/raw", force_refresh=True)["session_id"])
browse_set = set(query_cached(
    "SELECT DISTINCT session_id FROM laplaptech.user_event_tracking "
    "WHERE event_name = 'load_more_device_home' "
    "   OR (event_name = 'pageview' AND JSONExtractString(event_data, 'page_name') = 'Home')",
    cache_dir="../data/raw", force_refresh=True)["session_id"])

detail_without_search = detail_set - search_set
browse_and_no_search = detail_without_search & browse_set
pct_no_search = len(detail_without_search) / len(detail_set)
pct_confirmed = len(browse_and_no_search) / len(detail_without_search)

print(f"Xem chi tiết không qua tìm kiếm: {len(detail_without_search)} ({pct_no_search:.1%})")
print(f"  Trong đó xác nhận có duyệt qua Home/load_more: {pct_confirmed:.1%}")

funnel_final_note = f"""
## [Giai đoạn 5 - Trang 3] Kết luận chính thức về hiện tượng "Xem chi tiết" > "Tìm kiếm"

{len(detail_without_search)} session ({pct_no_search:.1%}) xem chi tiết KHÔNG qua tìm kiếm trước.
Trong nhóm này, {pct_confirmed:.1%} có bằng chứng đã duyệt qua Home/load_more_device_home —
MỘT PHẦN nguyên nhân, không phải toàn bộ (phần còn lại chưa xác định rõ nguồn gốc).

**Kết luận:** "Tìm kiếm -> Xem chi tiết -> Chọn So sánh" KHÔNG phải phễu chuyển đổi tuần tự bắt
buộc. Quyết định: Trang 3 dùng Bar chart "Mức độ tham gia theo từng hành động" thay vì Funnel
visual chuẩn (vốn ngụ ý quan hệ tập con giảm dần), kèm chú thích đúng mức độ bằng chứng.
"""
append_note(funnel_final_note)
print(funnel_final_note)

note_page3 = """
## [Giai đoạn 5 - Trang 3] Ghi chú công thức funnel

Đếm số session PHÂN BIỆT có event tương ứng, không bắt buộc thứ tự thời gian trong cùng session.
Endpoint đổi từ "click video review" (kế hoạch gốc) sang "select_device_for_comparison", do
không có dữ liệu tracking cho hành vi xem video (xác nhận ở Giai đoạn 3).
"""
append_note(note_page3)

Xem chi tiết không qua tìm kiếm: 16201 (36.7%)
  Trong đó xác nhận có duyệt qua Home/load_more: 54.1%

## [Giai đoạn 5 - Trang 3] Kết luận chính thức về hiện tượng "Xem chi tiết" > "Tìm kiếm"

16201 session (36.7%) xem chi tiết KHÔNG qua tìm kiếm trước.
Trong nhóm này, 54.1% có bằng chứng đã duyệt qua Home/load_more_device_home —
MỘT PHẦN nguyên nhân, không phải toàn bộ (phần còn lại chưa xác định rõ nguồn gốc).

**Kết luận:** "Tìm kiếm -> Xem chi tiết -> Chọn So sánh" KHÔNG phải phễu chuyển đổi tuần tự bắt
buộc. Quyết định: Trang 3 dùng Bar chart "Mức độ tham gia theo từng hành động" thay vì Funnel
visual chuẩn (vốn ngụ ý quan hệ tập con giảm dần), kèm chú thích đúng mức độ bằng chứng.



In [19]:
### [Giai đoạn 5 - Data Quality Monitor] Snapshot % null + anomaly theo thời gian
snapshot_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

df_benchmark_raw = query_cached("SELECT * FROM laplaptech.laptop_benchmark_result", cache_dir="../data/raw")
# Tự query lại laptop_model tại đây (không dựa vào biến df_laptop_raw từ cell Trang 2)
# để cell này chạy độc lập được, đúng nguyên tắc reproducibility.
df_laptop_for_quality = query_cached("SELECT * FROM laplaptech.laptop_model", cache_dir="../data/raw")
df_merged_quality = df_laptop_for_quality.merge(
    df_benchmark_raw, left_on="id", right_on="laptop_model_id", how="left", suffixes=("", "_bm")
)

cols_to_check = ["battery_capacity_whr", "laptop_weight", "cpu_tdp", "gpu_tdp",
                  "office_battery_result_minutes", "geekbench_6_cpu_multi_core_plugged_in"]
quality_rows = [
    {"snapshot_time": snapshot_time, "column": col,
     "null_pct": round(df_merged_quality[col].isna().mean(), 4)}
    for col in cols_to_check if col in df_merged_quality.columns
]
df_quality_snapshot = pd.DataFrame(quality_rows)
df_quality_snapshot["contaminated_device_count"] = len(detect_contaminated_devices(df_laptop_for_quality))
display(df_quality_snapshot)

quality_path = "../data/processed/dashboard_data_quality_history.csv"
# Tránh tích lũy nhiều snapshot trùng NGÀY khi chạy lại nhiều lần trong cùng 1 ngày để test
today_str = datetime.now().strftime("%Y-%m-%d")
if os.path.exists(quality_path):
    df_history = pd.read_csv(quality_path)
    df_history = df_history[~df_history["snapshot_time"].str.startswith(today_str)]
    df_combined = pd.concat([df_history, df_quality_snapshot], ignore_index=True)
else:
    df_combined = df_quality_snapshot
df_combined.to_csv(quality_path, index=False)
print(f"Đã cập nhật snapshot (thay thế snapshot cùng ngày nếu có) vào {quality_path}")

,snapshot_time,column,null_pct,contaminated_device_count
0,2026-09-20 13:09:15,battery_capacity_whr,0.0314,24
1,2026-09-20 13:09:15,laptop_weight,0.0314,24
2,2026-09-20 13:09:15,cpu_tdp,0.8868,24
3,2026-09-20 13:09:15,gpu_tdp,0.7862,24
4,2026-09-20 13:09:15,office_battery_result_minutes,0.9937,24
5,2026-09-20 13:09:15,geekbench_6_cpu_multi_core_plugged_in,0.9937,24


Đã cập nhật snapshot (thay thế snapshot cùng ngày nếu có) vào ../data/processed/dashboard_data_quality_history.csv


In [20]:
### [Giai đoạn 5] Data Dictionary cho Dashboard + quyết định refresh/schema
dashboard_dict = """
## [Giai đoạn 5] Data Dictionary cho Dashboard

| Metric hiển thị | Nguồn file | Công thức/Định nghĩa |
|---|---|---|
| Total Active Users | dashboard_overview_kpi.csv | COUNT(DISTINCT user_psuedo_id) |
| Total Sessions | dashboard_overview_kpi.csv | COUNT(DISTINCT session_id) |
| Interest Score | analysis_base.csv | 1*pageview(DeviceDetail) + 2*select_device_for_comparison |
| Battery Density | analysis_base.csv | battery_capacity_whr / laptop_weight_g |
| Golden Ratio Group | golden_ratio_laptops.csv | Top 25% battery_density + top 25% nhẹ nhất |
| Mức độ tham gia | dashboard_funnel_summary.csv | Số session phân biệt/hành động, KHÔNG bắt buộc thứ tự |
| Data Quality % Null | dashboard_data_quality_history.csv | Tỷ lệ NULL theo cột, 1 snapshot/ngày |

## [Giai đoạn 5] Quyết định Data Refresh & Schema Change

Refresh: chạy lại notebook 01->05, sau đó Refresh All trong Power BI. Không có lịch tự động.
Schema đổi: get_schema() ở notebook 01 sẽ báo lỗi sớm; detect_contaminated_devices() tự thích
ứng dữ liệu mới, không hardcode ID.
"""
append_note(dashboard_dict)

In [21]:
### [Giai đoạn 5] Tổng kết
stage5_summary = """
## [Giai đoạn 5] Tổng kết

Đã chuẩn bị đầy đủ dữ liệu nguồn cho Power BI: dashboard_overview_kpi.csv,
dashboard_segment_attention.csv, dashboard_cpu_battery_ratio.csv (thưa, chỉ để minh bạch),
dashboard_battery_density_by_cpu.csv (dùng chính), dashboard_funnel_summary.csv,
dashboard_data_quality_history.csv.

Phát hiện quan trọng trong quá trình chuẩn bị: "funnel" Tìm kiếm->Xem chi tiết->Chọn So sánh
KHÔNG phải phễu tuần tự (36.7% xem chi tiết không qua tìm kiếm), đã đổi cách trình bày trang 3
từ Funnel visual sang Bar chart "Mức độ tham gia", tránh gây hiểu lầm.

Output đã sẵn sàng để dựng laplaptech_dashboard.pbix (4 trang: Overview, Hardware Benchmark
Matrix, User Engagement, Data Quality Monitor). Sẵn sàng hoàn tất Giai đoạn 6 (6.3 One-pager,
6.4 Publish Dashboard) sau khi hoàn thành thao tác thủ công trên Power BI.
"""
append_note(stage5_summary)
print(stage5_summary)


## [Giai đoạn 5] Tổng kết

Đã chuẩn bị đầy đủ dữ liệu nguồn cho Power BI: dashboard_overview_kpi.csv,
dashboard_segment_attention.csv, dashboard_cpu_battery_ratio.csv (thưa, chỉ để minh bạch),
dashboard_battery_density_by_cpu.csv (dùng chính), dashboard_funnel_summary.csv,
dashboard_data_quality_history.csv.

Phát hiện quan trọng trong quá trình chuẩn bị: "funnel" Tìm kiếm->Xem chi tiết->Chọn So sánh
KHÔNG phải phễu tuần tự (36.7% xem chi tiết không qua tìm kiếm), đã đổi cách trình bày trang 3
từ Funnel visual sang Bar chart "Mức độ tham gia", tránh gây hiểu lầm.

Output đã sẵn sàng để dựng laplaptech_dashboard.pbix (4 trang: Overview, Hardware Benchmark
Matrix, User Engagement, Data Quality Monitor). Sẵn sàng hoàn tất Giai đoạn 6 (6.3 One-pager,
6.4 Publish Dashboard) sau khi hoàn thành thao tác thủ công trên Power BI.

